# Scoring de Crédit — Évaluation du Risque Bancaire
## Étape 1 : Analyse Exploratoire des Données (EDA)

Dans ce premier notebook, j'examine les données pour comprendre le profil des clients et identifier les signaux qui indiquent un risque de défaut de paiement. J'utilise deux sources principales : les informations de la demande de crédit actuelle et l'historique provenant du bureau de crédit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

# Configuration visuelle
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

### 1. Chargement et Dimensions
Je commence par charger les deux fichiers principaux pour voir l'ampleur des données que j'ai à traiter.

In [ ]:
app_train = pd.read_csv("../data/application_train.csv")
bureau = pd.read_csv("../data/bureau.csv")

print(f"Dimensions de application_train : {app_train.shape}")
print(f"Dimensions de bureau : {bureau.shape}")

### 2. Analyse du Risque (TARGET)
Je vérifie le taux de défaut de paiement global. C'est ma variable cible.

In [ ]:
taux_defaut = app_train['TARGET'].mean() * 100
print(f"Taux de défaut moyen : {taux_defaut:.2f}%")

# Visualisation du déséquilibre des classes
fig, ax = plt.subplots()
sns.countplot(x='TARGET', data=app_train, palette=['#2ecc71', '#e74c3c'])
plt.title("Déséquilibre des classes (0: OK, 1: Défaut)")
plt.show()

### 3. Facteurs de risque (Corrélations)
Je cherche les 10 variables numériques les plus liées au risque de défaut.

In [ ]:
correlations = app_train.select_dtypes(include=['number']).corr()['TARGET'].sort_values()
print("Top 5 corrélations négatives :")
print(correlations.head(5))
print("\nTop 5 corrélations positives :")
print(correlations.tail(6))

### 4. Distributions clés
J'observe comment se répartissent l'âge, les revenus et les montants des crédits.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Âge
app_train['AGE'] = app_train['DAYS_BIRTH'] / -365
sns.histplot(app_train['AGE'], bins=30, kde=True, ax=axes[0], color='blue')
axes[0].set_title("Distribution de l'Âge")

# Revenus (limité pour la lisibilité)
sns.histplot(app_train[app_train['AMT_INCOME_TOTAL'] < 500000]['AMT_INCOME_TOTAL'], bins=30, kde=True, ax=axes[1], color='green')
axes[1].set_title("Distribution des Revenus")

# Montant Crédit
sns.histplot(app_train['AMT_CREDIT'], bins=30, kde=True, ax=axes[2], color='purple')
axes[2].set_title("Distribution du Montant Crédit")

plt.tight_layout()
plt.show()

### 5. Jointure avec le Bureau
Je calcule le nombre de crédits précédents pour chaque client afin d'enrichir mon analyse.

In [ ]:
nb_credits = bureau.groupby('SK_ID_CURR').size().reset_index(name='NB_PREV_CREDITS')
app_train = app_train.merge(nb_credits, on='SK_ID_CURR', how='left')
app_train['NB_PREV_CREDITS'] = app_train['NB_PREV_CREDITS'].fillna(0)

print("Statistiques sur le nombre de crédits précédents :")
print(app_train['NB_PREV_CREDITS'].describe())

## Conclusions Business

1. **Déséquilibre massif** : Moins de 10% des clients sont en défaut. Je devrai utiliser des techniques de pondération pour que mon modèle ne néglige pas ces profils riskés.
2. **Importance des scores externes** : Les variables `EXT_SOURCE_1/2/3` sont les indicateurs les plus puissants. Elles sont cruciales pour la prédiction.
3. **L'âge est un facteur** : Je remarque que les clients les plus jeunes ont une fréquence de défaut plus élevée.
4. **Historique Bureau** : La jointure montre que les clients ont en moyenne 4 à 5 crédits précédents. Ce volume d'historique sera une mine d'or pour mon modèle.